# Retrieval Preparation
**ResolveX - M4: Evidence preparation and initial search experiments**

Prepare Ask Ubuntu cases and test BM25 (keyword search) and a small dense retrieval pilot (embedding search). Final comparison and held-out evaluation are still pending.

## 1. Load the prepared data

Reuse the M2 knowledge base and full duplicate links. Keep generated data, indexes and models locally under `data/processed/`.

In [4]:
from contextlib import closing
from pathlib import Path
import json
import sqlite3
import sys
import time

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "src/retrieval.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook within ResolveX")
sys.path.insert(0, str(ROOT))
from src.retrieval import (policy_review_sample, prepare_evidence, build_benchmark,
                           development_queries, build_bm25, evaluate_retrieval,
                           build_dense_pilot, file_hash)
print("Source: M2 knowledge_base.jsonl and full duplicate_relationships.csv")

Source: M2 knowledge_base.jsonl and full duplicate_relationships.csv


## 2. Review the answer-selection policy

Select one answer per question:

- Use the valid accepted answer if its score is zero or higher.
- Otherwise, use the highest positive-score answer; break ties by smallest answer ID.
- Exclude questions with no qualifying answer. Preserve the M2 data.

Read two examples from each of five policy groups, using development groups and seed 42. This AI-assisted review checks the rule, not technical correctness.

In [5]:
review = pd.DataFrame(policy_review_sample(ROOT))
display(review[["QuestionId", "Bucket", "Title", "InspectedAnswerId", "AnswerScore", "Selection", "SelectedAnswerId"]])
# Full text is available in review["QuestionText"] and review["AnswerText"].
print("Reviewed examples:", len(review))

,QuestionId,Bucket,Title,InspectedAnswerId,AnswerScore,Selection,SelectedAnswerId
0,369099,accepted_negative,Unable to download Ubuntu SDK,369107,-1,no_supported_answer,None
1,425018,accepted_negative,Log Out problem after upgrading from 13.04 to ...,425172,-1,no_supported_answer,None
2,94902,accepted_positive,How can I switch between keyboard layouts by p...,94910,56,accepted,94910
3,961991,accepted_positive,How shall I install and invoke IDLE for Python3?,961994,6,accepted,961994
4,500966,accepted_zero,Shell script to run VM,500970,0,accepted,500970
5,183885,accepted_zero,"Can't install Ubuntu on HP Series 2000, blank ...",183892,0,accepted,183892
6,567454,no_supported_answer,Install Ubuntu 14.04 alongside Windows 8 Problem,568458,0,no_supported_answer,None
7,251074,no_supported_answer,Uninstall Ubuntu 12.10 and revert back to Wind...,251076,0,no_supported_answer,None
8,44566,positive_fallback,Enable wifi on my computer - firmware missing,44574,1,positive_fallback,44574
9,258521,positive_fallback,Unable to install VirtualBox on Ubuntu 12.10: ...,260945,10,positive_fallback,260945


Reviewed examples: 10


### Review findings

The full questions and inspected answers were read for all ten examples.

| Question IDs | Finding |
| --- | --- |
| 94902, 961991 | Positive accepted answers address the requests, with version-specific instructions. |
| 500966, 183885 | Zero-score accepted answers qualify, but include VM state changes and old Ubuntu advice. |
| 369099, 425018 | Negative accepted answers are excluded: lock-removal commands and a plausible historical fix. Filtering may discard useful advice. |
| 44566, 258521 | Positive fallbacks offer troubleshooting steps; the first provides device discovery rather than a confirmed firmware fix. |
| 567454, 251074 | No accepted or positive-score answer qualifies. |

Acceptance and votes do not guarantee solved cases, safe instructions or current applicability.

## 3. Prepare the retrieval data

Use SQLite, a local database file, to store selected cases and their IDs, dates, scores, tags, links, licenses and author details. Resolve names from `Users.xml`, then available post display names; leave unavailable names missing.

Create chunks of about 180 words. Preserve all characters through stored offsets and keep code blocks whole, including oversized blocks. Smaller embedding windows do not change the saved text.

In [6]:
db_path, corpus = prepare_evidence(ROOT)
db_path = Path(db_path).resolve(strict=True)
print("Database:", db_path)

with closing(sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)) as connection:
    assert connection.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
    assert connection.execute("SELECT count(*) FROM cases").fetchone()[0] == corpus["counts"]["cases"]
    assert connection.execute("SELECT count(*) FROM chunks u LEFT JOIN cases c USING(question_id) WHERE c.question_id IS NULL").fetchone()[0] == 0
    author_preview = pd.read_sql_query("""
        SELECT c.question_id,c.answer_id,c.selection,
               COALESCE(qa.name,qp.name) AS question_author,
               COALESCE(aa.name,ap.name) AS answer_author
        FROM cases c LEFT JOIN authors qa ON c.question_owner=qa.owner_id
        LEFT JOIN authors aa ON c.answer_owner=aa.owner_id
        LEFT JOIN post_authors qp ON c.question_id=qp.post_id
        LEFT JOIN post_authors ap ON c.answer_id=ap.post_id
        ORDER BY c.question_id LIMIT 5
    """, connection)

display(pd.Series(corpus["counts"], name="Count").to_frame())
display(author_preview)
print("Corpus fingerprint:", corpus["fingerprint"])
print("Database checks passed.")

Database: C:\Users\jc\Desktop\ResolveX\data\processed\retrieval\evidence.sqlite


,Count
source_questions,318537
accepted,135482
chunks,618330
oversized_code_chunks,15509
cases,240555
positive_fallback,105073
no_supported_answer,77982
question_user_name,234932
question_post_name,5622
question_unavailable_name,1


,question_id,answer_id,selection,question_author,answer_author
0,1,2,accepted,Nathan Osman,txwikinger
1,100,137,accepted,David Siegel,MarkovCh1
2,1000006,1000556,accepted,Jonathan Holvey,Jonathan Holvey
3,1000009,1000203,accepted,Jamie Hutber,egmont
4,1000012,1000018,accepted,Manos Kounelakis,Sadegh Ghasemi


Corpus fingerprint: b3334850751bb6d7244789378c59329249f60478c4f5ce6a12ea67fd9b6e77c5
Database checks passed.


## 4. Prepare the retrieval benchmark

Use recorded duplicate targets as known relevant results. Build connected groups from all duplicate links, then split groups 80/20 into development and held-out sets with seed 42. Row proportions depend on group sizes. Keep available targets and report exclusions.

Queries use title and body. Exclude direct target links and duplicate notices, then remove remaining Ask Ubuntu post URLs. Filtering may remove useful questions too.

Exclude all results belonging to the query question and count each retrieved question once. Duplicate links are incomplete relevance labels; they do not verify answer correctness.

In [7]:
benchmark = build_benchmark(ROOT, db_path, corpus["fingerprint"])
display(pd.Series(benchmark["counts"], name="Count").to_frame())
queries = development_queries(db_path, limit=100, seed=42)
assert len(queries) == 100
assert len({q["group_id"] for q in queries}) == len(queries)
assert all(q["split"] == "development" for q in queries)
print("Development sample:", len(queries), "queries from distinct groups")
print("Saved assignments:", (db_path.parent / "query_assignments.csv").relative_to(ROOT))
print("Held-out retrieval queries scored: 0")

,Count
source_pairs,41931
unavailable_target_pairs,593
queries_with_available_targets,37066
direct_target_link_queries,5379
development_queries,28721
eligible_pairs,35654
heldout_queries,2924
duplicate_notice_queries,42
development_groups,6816
heldout_groups,1689


Development sample: 100 queries from distinct groups
Saved assignments: data\processed\retrieval\query_assignments.csv
Held-out retrieval queries scored: 0


## 5. Test keyword search with BM25

Index each case's title, question and selected answer. Preserve terms such as `apt-get`, `etc/fstab`, versions and IP addresses.

Start with k1=1.2, b=0.75 and positive log IDF. Evaluate 100 development queries from distinct groups using Recall@1/5/10 and MRR@10.

Reference: [BM25 settings](https://lucene.apache.org/core/9_12_1/core/org/apache/lucene/search/similarities/BM25Similarity.html).

In [8]:
started = time.perf_counter()
bm25 = build_bm25(db_path, corpus["fingerprint"])
bm25_build_seconds = time.perf_counter() - started
started = time.perf_counter()
bm25_rows = pd.DataFrame(evaluate_retrieval(queries, bm25.search))
bm25_seconds = time.perf_counter() - started
bm25_metrics = bm25_rows[["Recall@1", "Recall@5", "Recall@10", "MRR@10"]].mean()
display(bm25_metrics.to_frame("BM25 development"))
print("Indexed cases:", len(bm25.question_ids))
print("Vocabulary:", len(bm25.vectorizer.vocabulary_))
print("Build/load seconds:", round(bm25_build_seconds, 2))
print("Mean query milliseconds:", round(1000 * bm25_seconds / len(queries), 2))
print("Index bytes:", (db_path.parent / "bm25.joblib").stat().st_size)
print("Held-out retrieval queries scored: 0")

,BM25 development
Recall@1,0.055000
Recall@5,0.195000
Recall@10,0.240000
MRR@10,0.119813


Indexed cases: 240555
Vocabulary: 1531729
Build/load seconds: 11.01
Mean query milliseconds: 43.62
Index bytes: 193105582
Held-out retrieval queries scored: 0


## 6. Test embedding search on a small sample

Use pretrained `all-MiniLM-L6-v2` on CPU, with a fixed revision, and FAISS for exact cosine search over normalized vectors. No fine-tuning.

Randomly select 2,000 cases with seed 42, independently of relevance. Use overlapping embedding windows within 256 tokens, preserving stored code. Rank questions by their best window and exclude every window of the query itself.

Evaluate up to 100 development queries with targets in this sample. **Different candidate sets and queries prevent direct comparison with BM25.** The next comparison needs shared data.

Model: [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).

In [9]:
import torch
import faiss
from sentence_transformers import SentenceTransformer

torch.set_num_threads(4)
faiss.omp_set_num_threads(4)
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION,
    cache_folder=str(ROOT / "data/processed/embedding_cache"), device="cpu", local_files_only=True)
started = time.perf_counter()
dense, pilot = build_dense_pilot(db_path, corpus["fingerprint"], model, MODEL_ID, MODEL_REVISION,
                                limit=2000, seed=42)
pilot_build_seconds = time.perf_counter() - started
pilot_queries = development_queries(db_path, limit=100, seed=42, candidate_ids=pilot["candidate_ids"])
assert pilot_queries and all(q["split"] == "development" for q in pilot_queries)
started = time.perf_counter()
dense_rows = pd.DataFrame(evaluate_retrieval(pilot_queries, dense.search))
dense_seconds = time.perf_counter() - started
dense_metrics = dense_rows[["Recall@1", "Recall@5", "Recall@10", "MRR@10"]].mean()
display(dense_metrics.to_frame("Dense pilot development"))
print("Pilot cases:", pilot["cases"], "/", corpus["counts"]["cases"])
print("Embedding views:", pilot["embedding_views"], "dimensions:", pilot["dimension"])
print("Model token limit:", model.max_seq_length)
print("Pilot development queries:", len(pilot_queries))
print("Build/load seconds:", round(pilot_build_seconds, 2))
print("Mean query milliseconds:", round(1000 * dense_seconds / len(pilot_queries), 2))
print("Index bytes:", pilot["index_bytes"])
print("Held-out retrieval queries scored: 0")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,Dense pilot development
Recall@1,0.821429
Recall@5,0.928571
Recall@10,0.928571
MRR@10,0.872024


Pilot cases: 2000 / 240555
Embedding views: 7725 dimensions: 384
Model token limit: 256
Pilot development queries: 56
Build/load seconds: 0.04
Mean query milliseconds: 69.88
Index bytes: 11865645
Held-out retrieval queries scored: 0


## Findings and next steps

- **240,555 cases:** 135,482 accepted answers and 105,073 positive fallbacks; 77,982 questions excluded.
- **618,330 chunks**, including 15,509 oversized code chunks preserved whole. One question and one selected answer have unavailable author names.
- **28,721 development / 2,924 held-out queries.** Duplicate groups stay separate; no identical normalized query text crosses sets.

| Experiment | Candidate cases | Development queries | Recall@10 | MRR@10 |
| --- | ---: | ---: | ---: | ---: |
| BM25 | 240,555 | 100 | 0.2400 | 0.1198 |
| Dense pilot | 2,000 | 56 | 0.9286 | 0.8720 |

BM25 Recall@1 is **0.0550** and Recall@5 is **0.1950**. It misses many known targets. The dense pilot uses 7,725 windows with 384 dimensions. Its results use a smaller corpus and different queries, so they do not show whether it outperforms BM25.

Next: choose a shared corpus and queries that fit available resources, compare BM25/dense/hybrid, and inspect failures, source age and weak evidence. Then freeze settings, evaluate held-out queries and finish the reusable retrieval function.